In [1]:
from model.Model import model_MSG
from model.Plotter import plotter_MSG
from model.coders_linkage import CODERS_pull
from pstats import SortKey, Stats

import os
import sys

# Autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all


In [2]:

# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

%reload_ext autoreload
# Set-up
# Sys and OS Operations
os.chdir("../../")
msg = model_MSG(scenario_name='baseline')
platform, msg_scenario, msg_folder = msg.setup_scenario()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
2024-11-13 14:22:08,310  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2024-11-13 14:22:08,315  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/muhammadawais/.local/share/ixmp/localdb/default' (user: ixmp)...
Config file read successfully!
####################################################################################################
Config file 'model/scenarios/baseline/config.toml' passed format verification!
####################################################################################################

		Model creation time: 5 Min and 27 Sec 		
--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 11/13/24 14:27:59 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined


2024-11-13 14:28:38,809 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2024-11-13 14:28:38,814 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!



				Solving process time: 1 Min and 7 Sec				


In [12]:
msg_scenario.has_solution()

False

In [3]:
df = msg_scenario.var('EMISS')

In [7]:
nodes = 'World'

In [4]:
df_emis = df[(df['emission'] == 'TCE') & (df['type_tec'] == 'all') & ~(df['node'].str.contains('World'))]

In [5]:
df_emis.to_csv(msg_folder/'emissions_baseline.csv')

In [6]:
import pandas as pd

In [7]:
df_emis = pd.read_csv(msg_folder/'emissions_baseline.csv')

In [16]:
emissions_df = df_emis.copy()


# Define the exact emissions trajectory for Canada (in MtCO2eq)
canada_trajectory = {
    2020: 659e6,   # 659 MtCO2eq
    2025: 549e6,   # 549 MtCO2eq
    2030: 439e6,   # 439 MtCO2eq
    2035: 329e6,   # 329 MtCO2eq
    2040: 220e6,   # 220 MtCO2eq
    2045: 110e6,   # 110 MtCO2eq
    2050: 50e6,    # 50,000 MtCO2eq (likely meant to be 50 MtCO2eq)
    2055: 40e6,    # 40,000 MtCO2eq (likely meant to be 40 MtCO2eq)
    2060: 30e6     # 30,000 MtCO2eq (likely meant to be 30 MtCO2eq)
}

# Extract the unique provinces (nodes) excluding Canada
provinces = emissions_df['node'].unique()
provinces = provinces[provinces != 'Canada']

# Initialize a list to store the emission trajectories for each province
emission_trajectories_list = []

# Iterate over each province and apply the reduction schedule
for province in provinces:
    # Extract the emission data for the province
    province_data = emissions_df[(emissions_df['node'] == province)]
    
    # Create a dictionary to store the year and emission values
    province_emissions = {'province': [], 'year': [], 'emission': []}
    
    # Get the 2020 base year emission for the province
    base_year_emission = province_data[province_data['year'] == 2020]['lvl'].values[0]
    
    # Get the total emissions for Canada in 2020
    canada_2020_emission = canada_trajectory[2020]
    
    # Calculate the proportion of this province's emissions relative to Canada's total in 2020
    province_proportion = base_year_emission / canada_2020_emission
    
    # Iterate over the Canada trajectory and calculate the emissions for this province
    for year, canada_target in canada_trajectory.items():
        adjusted_emission = province_proportion * canada_target
        
        # Store the emission value in the dictionary
        province_emissions['province'].append(province)
        province_emissions['year'].append(year)
        province_emissions['emission'].append(adjusted_emission)
    
    # Convert the dictionary to a DataFrame and append to the list
    province_emissions_df = pd.DataFrame(province_emissions)
    emission_trajectories_list.append(province_emissions_df)

# Concatenate all the DataFrames into a single DataFrame
emission_trajectories_df = pd.concat(emission_trajectories_list).reset_index(drop=True)

df_bound = emission_trajectories_df.copy()
df_bound.rename(columns={'province':'node','year':'type_year','emission':'value'}, inplace=True)
df_bound['type_emission'] = 'TCE'
df_bound['type_tec'] = 'all'
df_bound['unit'] = '-'



msg_scenario.add_par('bound_emission', df_bound)
msg_scenario.commit('emissions bounded')


In [80]:
msg.commit('')

In [14]:
msg_scenario.remove_solution()
msg_scenario.check_out()
# msg.add_par('demand', df)

In [15]:
msg_scenario.par('bound_emission')

,node,type_emission,type_tec,type_year,value,unit


In [82]:
msg.remove_par('bound_emission', df_bound)

In [84]:
df_bound['value'] = df_bound['value'] / 4

In [21]:
msg_scenario.add_par('bound_emission', df_bound)

In [39]:
df = msg.par('growth_activity_lo')

In [40]:
tecs = ['loil_trp', 'loil_fs', 'loil_i', 'loil_rc', 'foil_fs','oil_extr_6','gas_fs']
df_selected = df[df['technology'].isin(tecs)]
df_selected.loc[df_selected['technology'].isin(tecs), 'value'] *= 2

In [78]:
msg.check_out()

In [30]:
msg.add_par('growth_activity_lo', df_selected)

In [17]:
msg.remove_solution()

In [20]:
path

PosixPath('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/netzero')

In [22]:
import pandas as pd

df = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/netzero/tax_emission.csv')

In [23]:
df

,node,type_emission,type_tec,type_year,value
0,Alberta,TCE,all,2025,165.425062
1,Alberta,TCE,all,2030,285.215625
2,Alberta,TCE,all,2035,387.893250
3,Alberta,TCE,all,2040,387.893250
4,Alberta,TCE,all,2045,387.893250
...,...,...,...,...,...
99,Yukon,TCE,all,2040,387.893250
100,Yukon,TCE,all,2045,387.893250
101,Yukon,TCE,all,2050,387.893250
102,Yukon,TCE,all,2055,387.893250


In [26]:
msg_scenario.add_par('tax_emission', df)

In [25]:
msg_scenario.check_out()

In [ ]:
df = make_df('tax_emission', 
             type_emission='TCE',
             type_tec='all',
             type_year='all',
             )

In [ ]:
2025	148.25
2024	$182.18
2025	$215.80
2026	$249.63
2027	$283.56
2028	$317.18
2029	$384.94
2030-2050	$384.94

In [ ]:
years = get_optimization_years(scen)
df = (
    scen.par("duration_period", filters={"year": years})
    .drop(["unit"], axis=1)
    .rename(columns={"value": "duration"})
    .set_index(["year"])
)
for yr in years:
    if years.index(yr) == 0:
        val = price * conversion_factor
    else:
        val = (
            df.loc[years[years.index(yr) - 1]].value
            * (1 + scen.par("drate").value.unique()[0]) ** df.loc[yr].duration
        )
    df.at[yr, "value"] = val
df = (
    df.reset_index()
    .drop(["duration"], axis=1)
    .rename(columns={"year": "type_year"})
)
df.loc[:, "node"] = "World"
df.loc[:, "type_emission"] = "TCE"
df.loc[:, "type_tec"] = "all"
df.loc[:, "unit"] = "USD/tC"

scen.check_out()
scen.add_par("tax_emission", df)
scen.commit("Added carbon price")
log.info(df.to_string())

,node,type_emission,type_tec,year,value
0,Alberta,TCE,all,2025,59.2
1,Alberta,TCE,all,2030,103.6
2,Alberta,TCE,all,2035,125.8
3,Alberta,TCE,all,2040,125.8
4,Alberta,TCE,all,2045,125.8
...,...,...,...,...,...
99,Yukon,TCE,all,2040,125.8
100,Yukon,TCE,all,2045,125.8
101,Yukon,TCE,all,2050,125.8
102,Yukon,TCE,all,2055,125.8


In [63]:
df['type_year'] = df['year']

In [29]:
msg.add_par('tax_emission', df)

In [30]:
msg.remove_par('bound_emission',df)

In [37]:
df = msg_scenario.par('bound_activity_lo')

In [38]:
tecs = ['GDP', 'Population', 'CH4_IndNonEnergyEM', 'N2O_OtherAgSo', 'CH4_IndNonEnergyEM', 'N2O_ONonAgSo', 'WasteDriver', 'CF4_Semico']

In [44]:
msg_scenario.remove_par('bound_activity_lo', df)

In [45]:
msg_scenario.add_par('bound_activity_lo', df_remove)

In [48]:
msg_scenario.remove_solution()

In [49]:
msg_scenario.check_out()

In [50]:
df = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/netzero/updated_obps_NetZero_fix_cost.csv')

In [52]:
msg_scenario.add_par('fix_cost', df)

In [41]:
df_remove = df[~df['technology'].isin(tecs)]

In [53]:
msg_scenario.commit('update tax and emission bound')

In [54]:
caseName = msg_scenario.model + '__' + msg_scenario.scenario + '__v' + str(msg_scenario.version)
msg_scenario.solve(solve_options={'barcrossalg': '2','lpmethod':'1'}, case=caseName)

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 11/13/24 18:40:09 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__baseline__v15.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__baseline__v15.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__baseline__v15.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
  

2024-11-13 18:40:52,393 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2024-11-13 18:40:52,395 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [ ]:
msg_scenario.to_excel(f"model_default_netzero_v1.xlsx")

In [51]:
df = msg.var("DEMAND").rename(columns={"lvl": "value"}).assign(unit="GWa")

In [29]:
df = msg_scenario.par('demand')

In [ ]:


data = msg_scenario.par('demand')
# Loop through each unique combination of node and commodity
nodes = data['node'].unique()
commodities = data['commodity'].unique()

# Define the reduction factors for the key years
reduction_factors = {
        2020: 1.0,    # No reduction in 2020
        2030: 0.94,   # 6% reduction by 2030
        2040: 0.87,    # 13% reduction by 2040
        2050: 0.78,   # 18% reduction by 2050
        2060: 0.8    # 20% reduction by 2060
    }

# Create a function to calculate the reduction factor for any given year
def calculate_reduction_factor(year):
    if year in reduction_factors:
        return reduction_factors[year]
    else:
        # Linear interpolation
        if year < 2050:
            return 1.0 + (0.82 - 1.0) * (year - 2020) / (2050 - 2020)
        else:
            return 0.82 + (0.80 - 0.82) * (year - 2050) / (2060 - 2050)

# Function to apply reduction
def apply_reduction(group):
    for index, row in group.iterrows():
        year = row['year']
        reduction_factor = calculate_reduction_factor(year)
        group.at[index, 'value'] = row['value'] * reduction_factor
    return group

# Apply the reduction to each combination of node and commodity
for node in nodes:
    for commodity in commodities:
        mask = (data['node'] == node) & (data['commodity'] == commodity)
        data.loc[mask] = apply_reduction(data.loc[mask])





,node,commodity,level,year,time,value,unit
0,NewfoundlandandLabrador,i_feed,useful,2020,year,0.908803,GWa
1,NewfoundlandandLabrador,i_feed,useful,2025,year,0.927879,GWa
2,NewfoundlandandLabrador,i_feed,useful,2030,year,0.944069,GWa
3,NewfoundlandandLabrador,i_feed,useful,2035,year,0.955454,GWa
4,NewfoundlandandLabrador,i_feed,useful,2040,year,0.953956,GWa
5,NewfoundlandandLabrador,i_feed,useful,2045,year,0.969390,GWa
6,NewfoundlandandLabrador,i_feed,useful,2050,year,0.923446,GWa
7,NewfoundlandandLabrador,i_feed,useful,2055,year,0.995671,GWa
8,NewfoundlandandLabrador,i_feed,useful,2060,year,1.021204,GWa
9,PrinceEdwardIsland,i_feed,useful,2020,year,0.042174,GWa


In [35]:
msg_scenario.add_par('demand', data)

In [34]:
msg_scenario.check_out()

In [36]:
df.to_csv('demands_new.csv')

In [52]:
df_new = msg.par("demand")

In [59]:
df_new[(df_new['node'] =='Alberta') & (df['commodity'] == 'transport')]

,node,commodity,level,year,time,value,unit
540,Alberta,rc_therm,useful,2020,year,14.672228,GWa
541,Alberta,rc_therm,useful,2025,year,17.405320,GWa
542,Alberta,rc_therm,useful,2030,year,20.515764,GWa
543,Alberta,rc_therm,useful,2035,year,23.531199,GWa
544,Alberta,rc_therm,useful,2040,year,26.867014,GWa
545,Alberta,rc_therm,useful,2045,year,30.579487,GWa
546,Alberta,rc_therm,useful,2050,year,34.564266,GWa
547,Alberta,rc_therm,useful,2055,year,38.961146,GWa
548,Alberta,rc_therm,useful,2060,year,43.550800,GWa


In [60]:
df[(df['node'] =='Alberta') & (df['commodity'] == 'transport')]

,node,commodity,level,year,time,value,mrg,unit
0,Alberta,transport,useful,2020,year,13.134694,0.0,GWa
1,Alberta,transport,useful,2025,year,16.032357,0.0,GWa
2,Alberta,transport,useful,2030,year,17.398884,0.0,GWa
3,Alberta,transport,useful,2035,year,19.330618,0.0,GWa
4,Alberta,transport,useful,2040,year,21.379482,0.0,GWa
5,Alberta,transport,useful,2045,year,23.795802,0.0,GWa
6,Alberta,transport,useful,2050,year,26.416787,0.0,GWa
7,Alberta,transport,useful,2055,year,29.254100,0.0,GWa
8,Alberta,transport,useful,2060,year,32.134819,0.0,GWa


In [53]:
is_different = not df_new.equals(df[['node', 'commodity', 'level', 'year', 'time', 'value', 'unit']])

In [54]:
is_different

True

In [43]:
df_new.equals(df[['node', 'commodity', 'level', 'year', 'time', 'value', 'unit']])

False

In [41]:
differences

,node,commodity,level,year,time,value,unit


In [39]:
is_different

True

In [ ]:

scen.check_out()
scen.add_par("demand", df)
scen.commit("parameter demand updated from source variable")

In [31]:
import ixmp 


mp = ixmp.Platform(name='local')


In [32]:
import numpy as np
import pandas as pd
import pyam
from message_ix import Reporter, Scenario
from typing import Optional, Tuple

from message_ix_models.report.legacy.iamc_report_hackathon import (
    report as legacy_report,
)
from message_ix_models.util import package_data_path

log = logging.getLogger(__name__)


def run_old_reporting(sc: Optional[Scenario] = None):
    if sc is None:
        raise ValueError("Must provide a Scenario object!")
    # mp2 = sc.platform

    log.info(
        " Start reporting of the global energy system (old reporting scheme)"
        f"for the scenario {sc.model}.{sc.scenario}"
    )
    legacy_report(
        mp=mp,
        scen=sc,
        merge_hist=True,
        merge_ts=False,
    )

In [29]:



Platform

ixmp4.core.platform.Platform

In [24]:
import logging

In [33]:
run_old_reporting(msg_scenario)

IndexError: list index out of range

In [35]:
msg_scenario.has_solution()

False

In [36]:
import message_ix

In [37]:
rep = message_ix.Reporter.from_scenario(
			msg_scenario,
			units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
		)
report = rep.get("message::default")


In [4]:
rep

In [1]:
from genno.compat.pyam import iamc as handle_iamc

In [3]:
rep.require_compat("message_ix_models.report.operator")

NameError: name 'rep' is not defined

In [ ]:
from message_ix_models.report import (
	broadcast,
	make_matched_dfs,
	same_node,
	same_time,
)

In [5]:
msg_scenario.par('demand')

NameError: name 'msg_scenario' is not defined

In [6]:
msg_scenario

NameError: name 'msg_scenario' is not defined